In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window


spark = (
    SparkSession.builder
        .appName("etl-datalake")
        .master("local[*]")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

26/04/02 14:25:34 WARN Utils: Your hostname, sarti-mint resolves to a loopback address: 127.0.1.1; using 192.168.1.107 instead (on interface enp3s0)
26/04/02 14:25:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 14:25:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/02 14:25:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/02 14:25:38 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
# Carregar os dados dos arquivos CSV para DataFrames do Spark - Camada de Ingestão - Raw Zone
df_rh_folha = spark.read \
    .option("header", True) \
    .csv("data/raw/rh_folha_pessoas")\
    .withColumn("origem_registro", F.lit("rh_folha"))

df_esocial = spark.read \
    .option("header", True) \
    .csv("data/raw/esocial_pessoas")\
    .withColumn("origem_registro", F.lit("esocial")) \

df_gestao = spark.read \
    .option("header", True) \
    .csv("data/raw/gestao_pessoas")\
    .withColumn("origem_registro", F.  lit("gestao")) 

In [3]:
# Visualizar os schemas dos DataFrames
df_rh_folha.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-01-01   |rh_folha       |
|55555555505|Ricardo Alves|Rua E, 500|2021-06-18   |5200   |2023-01-01   |rh_folha       |
|33333333303|Carlos Melo  |Rua C, 300|2018-11-20   |6000   |2023-01-01   |rh_folha       |
|22222222202|Maria Souza  |Rua B, 200|2019-03-15   |7000   |2023-01-01   |rh_folha       |
|11111111101|João Silva   |Rua A, 100|2020-01-10   |5000   |2023-01-01   |rh_folha       |
+-----------+-------------+----------+-------------+-------+-------------+---------------+



In [4]:
# Avaliar a quantidade de valores nulos em cada coluna do DataFrame
df_rh_folha.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in df_rh_folha.columns
]).show()

+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|cpf_nulls|nome_nulls|endereco_nulls|data_admissao_nulls|salario_nulls|data_registro_nulls|origem_registro_nulls|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|        0|         0|             0|                  0|            0|                  0|                    0|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+



In [5]:
# Avaliar a quantidade de valores duplicados com base na combinação de nome e data de admissão
df_rh_folha.groupBy("nome", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+----+-------------+-----+
|nome|data_admissao|count|
+----+-------------+-----+
+----+-------------+-----+



In [6]:
# Retirando os registros duplicados, mantendo apenas o mais recente com base na data de registro
window_spec = Window.partitionBy("nome").orderBy(F.col("data_registro").desc())
df_ranked = df_rh_folha.withColumn(
    "rn",
    F.row_number().over(window_spec)
)
df_rh_folha = df_ranked.filter("rn = 1").drop("rn")
df_rh_folha.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|33333333303|Carlos Melo  |Rua C, 300|2018-11-20   |6000   |2023-01-01   |rh_folha       |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-01-01   |rh_folha       |
|11111111101|João Silva   |Rua A, 100|2020-01-10   |5000   |2023-01-01   |rh_folha       |
|22222222202|Maria Souza  |Rua B, 200|2019-03-15   |7000   |2023-01-01   |rh_folha       |
|55555555505|Ricardo Alves|Rua E, 500|2021-06-18   |5200   |2023-01-01   |rh_folha       |
+-----------+-------------+----------+-------------+-------+-------------+---------------+



In [7]:
#########################################################################################################################

In [8]:
df_esocial.show(5, False)

+-----------+--------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome          |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+--------------+----------+-------------+-------+-------------+---------------+
|77777777707|Bruno Costa   |Rua G, 700|2020-09-10   |3900   |2023-03-01   |esocial        |
|87878787802|Amanda Letícia|Rua H, 800|2019-12-04   |5300   |2023-03-01   |esocial        |
|66666666606|Ana Lima      |Rua F, 600|2021-07-01   |4000   |2023-03-01   |esocial        |
|33333333303|Carlos Melo   |Rua C, 305|2018-11-20   |6100   |2023-03-01   |esocial        |
|44444444404|Fernanda Lima |Rua D, 400|2022-02-05   |4500   |2023-03-01   |esocial        |
+-----------+--------------+----------+-------------+-------+-------------+---------------+
only showing top 5 rows



In [9]:
# Avaliar a quantidade de valores nulos em cada coluna do DataFrame
df_esocial.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in df_esocial.columns
]).show()

+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|cpf_nulls|nome_nulls|endereco_nulls|data_admissao_nulls|salario_nulls|data_registro_nulls|origem_registro_nulls|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|        0|         0|             0|                  0|            0|                  0|                    0|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+



In [10]:
# Avaliar a quantidade de valores duplicados com base na combinação de nome e data de admissão
df_esocial.groupBy("nome", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+----+-------------+-----+
|nome|data_admissao|count|
+----+-------------+-----+
+----+-------------+-----+



In [11]:
# Retirando os registros duplicados, mantendo apenas o mais recente com base na data de registro
window_spec = Window.partitionBy("cpf").orderBy(F.col("data_registro").desc())
df_ranked = df_esocial.withColumn(
    "rn",
    F.row_number().over(window_spec)
)
df_esocial = df_ranked.filter("rn = 1").drop("rn")
df_esocial.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|11111111101|João Silva   |Rua A, 100|2020-01-10   |4800   |2023-03-01   |esocial        |
|33333333303|Carlos Melo  |Rua C, 305|2018-11-20   |6100   |2023-03-01   |esocial        |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-03-01   |esocial        |
|66666666606|Ana Lima     |Rua F, 600|2021-07-01   |4000   |2023-03-01   |esocial        |
|77777777707|Bruno Costa  |Rua G, 700|2020-09-10   |3900   |2023-03-01   |esocial        |
+-----------+-------------+----------+-------------+-------+-------------+---------------+
only showing top 5 rows



In [12]:
df_gestao.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|88888888808|Juliana Rocha|Rua H, 800|2019-12-01   |5300   |2023-05-01   |gestao         |
|55555555505|Ricardo Alves|Rua E, 550|2021-06-18   |5000   |2023-05-01   |gestao         |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4700   |2023-05-01   |gestao         |
|11111111101|João Silva   |Rua A, 120|2020-01-10   |5100   |2023-05-01   |gestao         |
|33333333303|Carlos Melo  |Rua C, 305|2018-11-20   |6000   |2023-05-01   |gestao         |
+-----------+-------------+----------+-------------+-------+-------------+---------------+



In [13]:
# Avaliar a quantidade de valores nulos em cada coluna do DataFrame
df_gestao.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in df_gestao.columns
]).show()

+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|cpf_nulls|nome_nulls|endereco_nulls|data_admissao_nulls|salario_nulls|data_registro_nulls|origem_registro_nulls|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|        0|         0|             0|                  0|            0|                  0|                    0|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+



In [14]:
# Avaliar a quantidade de valores duplicados com base na combinação de nome e data de admissão
df_gestao.groupBy("cpf", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+---+-------------+-----+
|cpf|data_admissao|count|
+---+-------------+-----+
+---+-------------+-----+



In [15]:
# Retirando os registros duplicados, mantendo apenas o mais recente com base na data de registro
window_spec = Window.partitionBy("cpf").orderBy(F.col("data_registro").desc())
df_ranked = df_gestao.withColumn(
    "rn",
    F.row_number().over(window_spec)
)
df_gestao = df_ranked.filter("rn = 1").drop("rn")
df_gestao.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|11111111101|João Silva   |Rua A, 120|2020-01-10   |5100   |2023-05-01   |gestao         |
|33333333303|Carlos Melo  |Rua C, 305|2018-11-20   |6000   |2023-05-01   |gestao         |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4700   |2023-05-01   |gestao         |
|55555555505|Ricardo Alves|Rua E, 550|2021-06-18   |5000   |2023-05-01   |gestao         |
|88888888808|Juliana Rocha|Rua H, 800|2019-12-01   |5300   |2023-05-01   |gestao         |
+-----------+-------------+----------+-------------+-------+-------------+---------------+



In [16]:
df_rh_folha.printSchema()

root
 |-- cpf: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- endereco: string (nullable = true)
 |-- data_admissao: string (nullable = true)
 |-- salario: string (nullable = true)
 |-- data_registro: string (nullable = true)
 |-- origem_registro: string (nullable = false)



In [17]:
df_gestao.printSchema()


root
 |-- cpf: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- endereco: string (nullable = true)
 |-- data_admissao: string (nullable = true)
 |-- salario: string (nullable = true)
 |-- data_registro: string (nullable = true)
 |-- origem_registro: string (nullable = false)



In [18]:
df_esocial.printSchema()


root
 |-- cpf: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- endereco: string (nullable = true)
 |-- data_admissao: string (nullable = true)
 |-- salario: string (nullable = true)
 |-- data_registro: string (nullable = true)
 |-- origem_registro: string (nullable = false)



In [19]:
df = df_rh_folha.unionByName(df_esocial).unionByName(df_gestao)
df.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|33333333303|Carlos Melo  |Rua C, 300|2018-11-20   |6000   |2023-01-01   |rh_folha       |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-01-01   |rh_folha       |
|11111111101|João Silva   |Rua A, 100|2020-01-10   |5000   |2023-01-01   |rh_folha       |
|22222222202|Maria Souza  |Rua B, 200|2019-03-15   |7000   |2023-01-01   |rh_folha       |
|55555555505|Ricardo Alves|Rua E, 500|2021-06-18   |5200   |2023-01-01   |rh_folha       |
+-----------+-------------+----------+-------------+-------+-------------+---------------+
only showing top 5 rows



In [20]:
# 🔹 1. Prioridade de fonte
df = df.withColumn(
    "source_priority",
    F.when(F.col("origem_registro") == "esocial", 1)
     .when(F.col("origem_registro") == "rh_folha", 2)
     .otherwise(3)
)

# 🔹 2. Hash robusto (com padronização)
df = df.withColumn(
    "hash_change",
    F.sha2(
        F.concat_ws("||",
            F.coalesce(F.lower(F.trim(F.col("nome"))), F.lit("")),
            F.coalesce(F.lower(F.trim(F.col("endereco"))), F.lit("")),
            F.coalesce(F.col("salario").cast("string"), F.lit(""))
        ),
        256
    )
)

# 🔹 3. Ordenação correta (data + prioridade)
window_hist = Window.partitionBy("cpf").orderBy(
    F.col("data_registro"),
    F.col("source_priority")
)

# 🔹 4. Hash anterior
df_hist = df.withColumn(
    "prev_hash",
    F.lag("hash_change").over(window_hist)
)

# 🔹 5. Detectar mudança real
df_hist = df_hist.withColumn(
    "mudou",
    F.when(
        (F.col("prev_hash").isNull()) | (F.col("hash_change") != F.col("prev_hash")),
        1
    ).otherwise(0)
)

# FILTRO CRÍTICO (remove histórico inútil)
df_hist = df_hist.filter(F.col("mudou") == 1)

# Criar vigência
window_hist2 = Window.partitionBy("cpf").orderBy("data_registro")

df_hist = df_hist.withColumn(
    "data_inicio",
    F.col("data_registro")
)

df_hist = df_hist.withColumn(
    "proxima_data",
    F.lead("data_registro").over(window_hist2)
)

df_hist = df_hist.withColumn(
    "data_fim",
    F.expr("date_sub(proxima_data, 1)")
)

# 🔹 8. Flag atual
df_hist = df_hist.withColumn(
    "is_current",
    F.when(F.col("proxima_data").isNull(), 1).otherwise(0)
)

# Criação do campo de data de ingestão - otimização de consultas futuras
df_final = df_hist.withColumn("data_ingestao", F.current_date())

In [21]:
df_final.show(5, False)

+-----------+-----------+----------+-------------+-------+-------------+---------------+---------------+----------------------------------------------------------------+----------------------------------------------------------------+-----+-----------+------------+----------+----------+-------------+
|cpf        |nome       |endereco  |data_admissao|salario|data_registro|origem_registro|source_priority|hash_change                                                     |prev_hash                                                       |mudou|data_inicio|proxima_data|data_fim  |is_current|data_ingestao|
+-----------+-----------+----------+-------------+-------+-------------+---------------+---------------+----------------------------------------------------------------+----------------------------------------------------------------+-----+-----------+------------+----------+----------+-------------+
|11111111101|João Silva |Rua A, 100|2020-01-10   |5000   |2023-01-01   |rh_folha       |2     

In [22]:
# Verificação final de registros e colunas
num_linhas = df_final.count()
num_colunas = len(df_final.columns)

print(f"Linhas: {num_linhas}")
print(f"Colunas: {num_colunas}")

Linhas: 15
Colunas: 16


In [23]:
# Avaliar a quantidade de valores duplicados com base na combinação de nome e data de admissão
df_final.groupBy("nome", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+-------------+-------------+-----+
|         nome|data_admissao|count|
+-------------+-------------+-----+
|Ricardo Alves|   2021-06-18|    2|
|   João Silva|   2020-01-10|    3|
|  Carlos Melo|   2018-11-20|    3|
|Fernanda Lima|   2022-02-05|    2|
+-------------+-------------+-----+



In [24]:
# Salvando em Parquet particionado por data ou is_current (boa prática)
(df_final
    .repartition(4)
    .write
    .mode("overwrite")
    .partitionBy("is_current")
    .format("parquet")
    .save("data_lake/dim_pessoa_historica")
)

In [25]:
# Conferindo os dados salvos - consulta otimizada para registros atuais
df_leitura = spark.read.parquet("data_lake/dim_pessoas")
df_leitura.createOrReplaceTempView("pessoas_view")

# Compatibilidade com SQL
spark.sql("""
    SELECT 
        nome, 
        salario, 
        origem_registro
    FROM pessoas_view 
    WHERE is_current = 1 
      AND data_ingestao = current_date()
""").show()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/sarti/Documentos/CaseEngenheiroDados-Volpy/data_lake/dim_pessoas.